In [1]:
import os
from ultralytics import YOLO

# Load trained model
model = YOLO("../train/runs/detect/runs/laboro_fast_high_map50/weights/best.pt")

# Paths
image_folder = "../data/infield/images"
label_folder = "../data/infield_labels"

# Create label folder if not exists
os.makedirs(label_folder, exist_ok=True)

# Run prediction
results = model.predict(
    source=image_folder,
    conf=0.4,
    save=False,   # no need to save images
    device=0
)

# Loop through results
for r in results:
    img_path = r.path
    img_name = os.path.basename(img_path)
    txt_name = os.path.splitext(img_name)[0] + ".txt"
    txt_path = os.path.join(label_folder, txt_name)

    boxes = r.boxes

    # Skip if no detections
    if boxes is None or len(boxes) == 0:
        open(txt_path, "w").close()  # empty file
        continue

    with open(txt_path, "w") as f:
        for box in boxes:
            cls = int(box.cls[0])
            xywh = box.xywhn[0].tolist()  # normalized (YOLO format)

            line = f"{cls} {' '.join(map(str, xywh))}\n"
            f.write(line)

print("✅ Labels generated in:", label_folder)


image 1/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (1).jpeg: 640x320 4 l_fully_ripeneds, 1 l_half_ripened, 3 l_greens, 101.7ms
image 2/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (10).jpeg: 640x320 4 l_fully_ripeneds, 21.1ms
image 3/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (100).jpeg: 640x320 (no detections), 16.9ms
image 4/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (101).jpeg: 640x320 (no detections), 20.5ms
image 5/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (102).jpeg: 640x320 (no detections), 16.0ms
image 6/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (103).jpeg: 640x320 (no detections), 17.6ms
image 7/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (104).jpeg: 640x320 (no detections), 17.3ms
image 8/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (105).jpeg: 640x320 (no detections), 15.3ms
image 9/123 d:\FYP\PlantGrowth\train\..\data\infield\images\sample (106).jpeg

In [ ]:
import os
import cv2

# Paths
image_folder = "../data/infield/images"
label_folder = "../data/infield/infield_labels"   # make sure this is correct!

# Class names
class_names = [
    "b_fully_ripened",
    "b_half_ripened",
    "b_green",
    "l_fully_ripened",
    "l_half_ripened",
    "l_green"
]

for img_name in os.listdir(image_folder):

    if not img_name.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    img_path = os.path.join(image_folder, img_name)
    label_path = os.path.join(label_folder, os.path.splitext(img_name)[0] + ".txt")

    image = cv2.imread(img_path)

    if image is None:
        print("❌ Cannot read:", img_path)
        continue

    h, w, _ = image.shape

    # Draw labels if exist
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            lines = f.readlines()

        for line in lines:
            cls, x, y, bw, bh = map(float, line.strip().split())
            cls = int(cls)

            # Convert YOLO → pixel coords
            x1 = int((x - bw/2) * w)
            y1 = int((y - bh/2) * h)
            x2 = int((x + bw/2) * w)
            y2 = int((y + bh/2) * h)

            # Draw box
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Label text
            cv2.putText(image, class_names[cls], (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    else:
        print(f"⚠️ No label for {img_name}")

    # Show image
    cv2.imshow("YOLO Visualization", image)
    cv2.namedWindow("YOLO Visualization", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("YOLO Visualization", 1000, 700)
    key = cv2.waitKey(0)  # wait for key press

    if key == 27:  # ESC to exit
        break

cv2.destroyAllWindows()